# 📈 SHAP Özellik Önem Analizi — 4-Hedefli DNN

Bu notebook, **SHAP (SHapley Additive exPlanations)** yöntemiyle  
derin öğrenme modelinin tahminlerini yorumlar.

**SHAP nedir?**  
Oyun teorisinden türetilen bir yöntemdir. Her özelliğin tahmine katkısını,  
diğer özelliklerle olan etkileşimler de hesaba katılarak *adil* şekilde dağıtır.  
Bu sayede "Model neden bu tahmini yaptı?" sorusuna cevap verilir.

**Kullanılan SHAP yöntemi:** `DeepExplainer`  
PyTorch modelleri için optimize edilmiş, gradyan tabanlı SHAP hesaplaması.

**Her hedef için ayrı SHAP analizi yapılır:**
1. `formation_energy_per_atom`
2. `band_gap`
3. `cbm`
4. `energy_above_hull`

**Gereksinimler:** `pip install shap`  
**Model:** `models/best_model.pth`, `models/normalization_stats.pth`

In [ ]:
# ============================================================
# KÜTÜPHANELER VE AYARLAR
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

try:
    import shap
except ImportError:
    raise ImportError(
        'SHAP kütüphanesi bulunamadı. '
        'Yüklemek için: pip install shap'
    )

plt.rcParams['figure.dpi'] = 120

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Cihaz: {device}')

In [ ]:
# ============================================================
# MODEL MİMARİSİ (train.py ile aynı olmalı)
# ============================================================

class NeuralNetwork(nn.Module):
    """4-çıkışlı Çok-Hedefli Derin Sinir Ağı"""
    def __init__(self, input_size, output_size):
        super(NeuralNetwork, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 512), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, 512),        nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, 256),        nn.ReLU(),
            nn.Linear(256, 128),        nn.ReLU(),
            nn.Linear(128, 64),         nn.ReLU(),
            nn.Linear(64, 32),          nn.ReLU(),
            nn.Linear(32, output_size)
        )

    def forward(self, x):
        return self.layers(x)

print('Model sınıfı tanımlandı.')

In [ ]:
# ============================================================
# VERİ VE MODEL YÜKLEME
# ============================================================

# Önişlenmiş özellik matrisi
X = pd.read_csv('data/X_preprocessed.csv')
print(f'X boyutu: {X.shape}')

# Normalizasyon istatistiklerini yükle
norm = torch.load('models/normalization_stats.pth', map_location=device)
X_mean       = norm['X_mean'].to(device)
X_std        = norm['X_std'].to(device)
target_names = norm['target_names']
n_targets    = norm['n_targets']

print(f'Hedef değişkenler: {target_names}')

# Modeli yükle
model = NeuralNetwork(X.shape[1], n_targets).to(device)
model.load_state_dict(torch.load('models/best_model.pth', map_location=device))
model.eval()

# X'i tensöre çevir ve normalize et
eps = 1e-8
X_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
X_norm   = (X_tensor - X_mean) / (X_std + eps)

print('Model ve veri başarıyla yüklendi.')

In [ ]:
# ============================================================
# SHAP DeepExplainer HESAPLAMASI
# ============================================================
# DeepExplainer, PyTorch modellerinde backpropagation temelli SHAP hesaplar.
# Büyük veri setlerinde tüm örnekler üzerinde SHAP çok yavaş olur.
# Bu nedenle:
#   - background: 100 örnek referans seti (SHAP baseline)
#   - subset   : 300 örnek SHAP hesaplama seti
# İstediğiniz sayıyı artırabilirsiniz (daha doğru sonuç, daha yavaş hesap).

np.random.seed(42)
N_TOTAL = len(X_norm)

# Rastgele örneklem seç
bg_idx  = np.random.choice(N_TOTAL, min(100, N_TOTAL), replace=False)
sub_idx = np.random.choice(N_TOTAL, min(300, N_TOTAL), replace=False)

background = X_norm[bg_idx]   # arkaplan örnekleri
subset     = X_norm[sub_idx]  # SHAP hesaplanacak örnekler

print(f'Arkaplan boyutu : {background.shape}')
print(f'Hesaplama seti  : {subset.shape}')
print('SHAP hesaplanıyor... (birkaç dakika sürebilir)')

# DeepExplainer başlat
explainer = shap.DeepExplainer(model, background)

# SHAP değerleri hesapla
# shap_values: liste (her hedef için bir dizi) ya da tek dizi
shap_values = explainer.shap_values(subset)

# 4 hedef için shap_values bir liste olabilir
# shap_values[i]: shape (n_subset, n_features) — i. hedef için
if not isinstance(shap_values, list):
    # Tek çıkışlı model durumuna karşı koruma
    shap_values = [shap_values]

# n_targets değerini doğrula
print(f'\nSHAP hesaplandı: {len(shap_values)} hedef × {shap_values[0].shape}')

In [ ]:
# ============================================================
# HER HEDEF İÇİN SHAP ÖZET GRAFİĞİ (Summary Plot)
# ============================================================
# Özet grafiği iki bilgiyi bir arada gösterir:
#   - Yatay eksen: SHAP değeri (özelliğin tahmine katkısı)
#   - Renk       : özelliğin gerçek değeri (kırmızı=yüksek, mavi=düşük)
#   - Dikey sıralama: en önemli özellik en üstte

X_subset_np = subset.detach().cpu().numpy()

for i, hedef in enumerate(target_names):
    if i >= len(shap_values):
        break
    sv_i = np.squeeze(shap_values[i])  # (n_subset, n_features)

    plt.figure(figsize=(9, 8))
    shap.summary_plot(
        sv_i, X_subset_np,
        feature_names=X.columns.tolist(),
        show=False,
        max_display=20  # en önemli 20 özelliği göster
    )
    plt.title(
        f'SHAP Özet Grafiği — {hedef}\n'
        f'(Kırmızı=Yüksek özellik değeri, Mavi=Düşük)',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(
        f'figures/shap_ozet_{hedef.replace("_","-")}.png',
        dpi=150, bbox_inches='tight'
    )
    plt.show()

In [ ]:
# ============================================================
# ORTALAMA MUTLAK SHAP DEĞERİ — En Önemli 20 Özellik
# ============================================================
# Özelliğin tüm örneklerdeki ortalama |SHAP| değeri,
# o özelliğin **global** önemini gösterir.

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
renkler = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
TOP_N = 20

shap_ozet = {}  # tüm hedeflerin özeti için kaydet

for i, (ax, hedef, renk) in enumerate(zip(axes.flatten(), target_names, renkler)):
    if i >= len(shap_values):
        break
    sv_i = np.squeeze(shap_values[i])
    ort_mutlak = np.abs(sv_i).mean(axis=0)  # her özellik için ortalama |SHAP|

    # Sırala ve top N al
    siralama   = np.argsort(ort_mutlak)[::-1]
    top_idx    = siralama[:TOP_N]
    top_deger  = ort_mutlak[top_idx]
    top_isim   = np.array(X.columns)[top_idx]

    # Önem değerlerini sakla
    shap_ozet[hedef] = pd.Series(ort_mutlak, index=X.columns)

    # Yatay çubuk grafiği
    ax.barh(top_isim[::-1], top_deger[::-1], color=renk, alpha=0.8, edgecolor='none')
    ax.set_xlabel('Ortalama |SHAP| Değeri', fontsize=10)
    ax.set_title(f'{hedef}\nEn Önemli {TOP_N} Özellik', fontsize=10, fontweight='bold')

plt.suptitle('SHAP Özellik Önemi — 4 Hedef Karşılaştırması',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/shap_bar_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# SONUÇLARI KAYDET — CSV
# ============================================================

# Her hedef için özellik önemi DataFrame oluştur
onem_df = pd.DataFrame(shap_ozet)
onem_df.index.name = 'ozellik'
onem_df.to_csv('shap_ozellik_onemi_4hedef.csv')

print('SHAP sonuçları \'shap_ozellik_onemi_4hedef.csv\' dosyasına kaydedildi.')
print('\nHer hedef için en önemli 10 özellik:')
print('=' * 70)
for hedef in target_names:
    if hedef in shap_ozet:
        top10 = shap_ozet[hedef].sort_values(ascending=False).head(10)
        print(f'\n[{hedef}]')
        for ozellik, deger in top10.items():
            print(f'  {ozellik:<30} {deger:.5f}')

## SHAP Analizi — Yorum Kılavuzu

### Özet Grafiğini Nasıl Okumalı?
- **Sıralama:** Üstteki özellikler genel olarak daha önemlidir
- **Renk:** Kırmızı → yüksek özellik değeri, Mavi → düşük özellik değeri  
- **Yatay konum:** Sağa yayılım → tahmini artırır, sola yayılım → tahmini azaltır

### Beklenen Bulgular

**Formasyon Enerjisi için:**  
Ağırlıklı ortalama atom kütlesi ve belirli element fraksiyonları (nadir toprak elementleri)  
öne çıkması beklenir. Yüksek elektron ilgili elementler (O, F) negatif formasyon enerjisine katkı sağlar.

**Band Gap için:**  
Elektronegativite istatistikleri (en_mean, en_range) ve belirli element fraksiyonları  
önemli olmalıdır. Yüksek en_range → büyük bant aralığı (iyonik bileşikler).

**Hull Üstü Enerji için:**  
Uzay grubu one-hot özellikleri ve element bileşimi kritik rol oynar.  
Yüksek simetri uzay grupları (cubic) genellikle daha kararlı malzemeler verir.

### DFT Sınırlılığı Notu
SHAP analizi, modelin GGA DFT verilerinden öğrendiği ilişkileri yansıtır.  
Geçiş metali oksitlerinde (Fe, Co, Ni içeren) bu ilişkiler sistematik hata içerebilir.  
Detaylar için `examine.ipynb` defterine bakınız.